# Как научить компьютер читать?

В этой тетрадке мы обучим свой собственный word2vec. Делать мы это будем на каком-нибудь не очень большом тексте, который вам предстоит выбрать самому. На выбор есть [несколько сказок](https://github.com/nevmenandr/word2vec-russian-novels/tree/master/vector-school) и других [литературных штук](https://github.com/nevmenandr/word2vec-russian-novels/tree/master/books_before) из школьной программы.

In [1]:
# Ссылка на выбранное вами произведение
# Для примера возьмем Преступление и Наказание
url = "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/CrimeAndPunishment.txt"

Спарсим текст из файлика.

In [2]:
import requests

resp = requests.get(url)
text = resp.text

# Последние 500 символов. Аккуратно! Спойлеры!
print(text[-500:])

смотреть на эти семь лет, как на семь
дней. Он даже и не знал того, что новая жизнь не даром же ему достается, что ее надо еще дорого купить, заплатить за нее великим, будущим подвигом...
Но тут уж начинается новая история, история постепенного обновления человека, история постепенного перерождения его, постепенного перехода из одного мира в другой, знакомства с новою, доселе совершенно неведомою действительностью. Это могло бы составить тему нового рассказа, - но теперешний рассказ наш окончен.


## 1. Предобработка

Теперь нам надо его немного предобработать.  Пусть все слова пишутся с маленькой буквы.

In [3]:
text = text.lower()

Разобьём весь текст на предложения.

In [4]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import sent_tokenize

sents = sent_tokenize(text)
len(sents)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


13702

In [5]:
sents[220]

'действительно, на его платье и даже в волосах кое-где виднелись прилипшие былинки сена.'

Разобьём каждое предложение на отдельные слова.

In [6]:
from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer("\w+")
tokenizer.tokenize(sents[220])

<>:3: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_11749/3296176897.py:3: SyntaxWarning: invalid escape sequence '\w'
  tokenizer = RegexpTokenizer("\w+")


['действительно',
 'на',
 'его',
 'платье',
 'и',
 'даже',
 'в',
 'волосах',
 'кое',
 'где',
 'виднелись',
 'прилипшие',
 'былинки',
 'сена']

In [7]:
sents_tokenize = [tokenizer.tokenize(sent) for sent in sents]
sents_tokenize[4:6]

[['и',
  'каждый',
  'раз',
  'молодой',
  'человек',
  'проходя',
  'мимо',
  'чувствовал',
  'какое',
  'то',
  'болезненное',
  'и',
  'трусливое',
  'ощущение',
  'которого',
  'стыдился',
  'и',
  'от',
  'которого',
  'морщился'],
 ['он',
  'был',
  'должен',
  'кругом',
  'хозяйке',
  'и',
  'боялся',
  'с',
  'нею',
  'встретиться']]

In [8]:
# Flatten без numpy :)
words = [item for sent in  sents_tokenize for item in sent]

In [9]:
len(words)  # всего слов

173403

In [10]:
len(set(words))  # уникальных слов

24925

Можно выбросить все стоп-слова.

In [11]:
from nltk.corpus import stopwords

stopwords_ru = stopwords.words("russian")
display(len(stopwords_ru))
stopwords_ru[:10]

151

['и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со']

In [12]:
sents_tokenize = [[word for word in text_cur if word not in stopwords_ru] for text_cur in sents_tokenize]

Слов в корпусе не очень много. Давайте лемматизируем их.  В этом нам поможет библиотека **pymorphy2.**

**pymorphy2** — это полноценный морфологический анализатор, целиком написанный на Python. Он также умеет ставить слова в нужную форму (спрягать и склонять). [Документация по pymorphy2.](https://pymorphy2.readthedocs.io/en/latest/)

In [13]:
!pip install pymorphy3 --quiet

In [14]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

text = "Филипп пошёл в Авеньон и пленил пап!"
tokens = tokenizer.tokenize(text)

" ".join(morph.normal_forms(token)[0] for token in tokens)

'филипп пойти в авеньон и пленить папа'

In [15]:
p = morph.parse("стали")
p

[Parse(word='стали', tag=OpencorporaTag('VERB,perf,intr plur,past,indc'), normal_form='стать', score=0.975342, methods_stack=((DictionaryAnalyzer(), 'стали', 945, 4),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn sing,gent'), normal_form='сталь', score=0.010958, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 1),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn plur,nomn'), normal_form='сталь', score=0.005479, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 6),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn sing,datv'), normal_form='сталь', score=0.002739, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 2),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn sing,loct'), normal_form='сталь', score=0.002739, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 5),)),
 Parse(word='стали', tag=OpencorporaTag('NOUN,inan,femn plur,accs'), normal_form='сталь', score=0.002739, methods_stack=((DictionaryAnalyzer(), 'стали', 13, 9),))]

Обработаем все слова из датасета.

In [16]:
sents_tokenize = [[morph.normal_forms(word)[0] for word in text_cur] for text_cur in sents_tokenize]

In [17]:
# Flatten без numpy :)
words = [item for sent in  sents_tokenize for item in sent]

In [18]:
len(words)  # всего слов

93069

In [19]:
len(set(words))  # уникальных слов

11083

Хватит обработок! Мы тут не анализом текстов занимаемся, а нейросетками. Если хотите больше предобработки, [можете прочитать этот мануал](https://nbviewer.jupyter.org/github/FUlyankin/hse_texts_do/blob/master/sem_1/texts_sem1.ipynb) об этом.  Давайте построим словарик с частотностями и перейдём к моделированию.

In [20]:
from collections import Counter

word_dict = Counter(words)
word_dict.most_common()[:20]

[('это', 1449),
 ('всё', 963),
 ('знать', 630),
 ('раскольник', 567),
 ('свой', 549),
 ('один', 548),
 ('сказать', 544),
 ('говорить', 536),
 ('человек', 501),
 ('весь', 442),
 ('стать', 441),
 ('мочь', 441),
 ('который', 430),
 ('сам', 430),
 ('такой', 395),
 ('очень', 387),
 ('какой', 379),
 ('соня', 379),
 ('рука', 369),
 ('петрович', 369)]

In [21]:
words = word_dict.most_common()
len([item for item in words if item[1] >= 3])  # совсем мало :)

4767

## 2. Моделирование

__Основные параметры:__

* данные должны быть итерируемым объектом
* `vector_size` — размер вектора,
* `window` — размер окна наблюдения,
* `min_count` — мин. частотность слова в корпусе,
* `sg` — используемый алгоритм обучения (0 — CBOW, 1 — Skip-gram),
* `sample` — порог для downsampling'a высокочастотных слов,
* `workers` — количество потоков,
* `alpha` — learning rate,
* `iter` — количество итераций,
* `max_vocab_size` — позволяет выставить ограничение по памяти при создании словаря (т.е. если ограничение привышается, то низкочастотные слова будут выбрасываться). Для сравнения: 10 млн слов = 1Гб RAM.

In [22]:
!pip install gensim --quiet

In [23]:
%%time
from gensim.models.word2vec import Word2Vec

# vector_size - размерность векторов, которые мы хотим обучить
# window - ширина окна контекста
# min_count - если слово встречается реже, для него не учим модель
model = Word2Vec(vector_size=100, window=2, min_count=3, workers=4)

# строительство словаря, чтобы обучение шло быстрее
model.build_vocab(sents_tokenize)

# обучение модели
# первый аргумент - наша выборка, генератор будет вкидывать в модель наши тексты, пока они не кончатся
# второй аргумент - число примеров в выборке
# третий аргумент - количество эпох обучения: сколько раз модель пройдётся по всему корпусу текстов
model.train(sents_tokenize, total_examples=model.corpus_count, epochs=100)

# !NB в ситуации, когда у нас огромный корпус, 100 эпох это слишком много!

CPU times: user 24.8 s, sys: 258 ms, total: 25.1 s
Wall time: 25.4 s


(7944247, 9306900)

In [24]:
model.corpus_count  # число примеров в обучающей выборке

13702

Смотрим, сколько в модели слов.

In [25]:
len(model.wv.key_to_index)

4767

In [26]:
"старуха" in model.wv.key_to_index

True

## 3. Свойства модели

In [27]:
# вектор слова
model.wv["старуха"][:10]

array([-0.52957153,  0.04244972,  0.7182297 , -0.23024775, -0.05635339,
       -0.36571303, -0.23705195, -0.5932071 ,  0.32038304, -0.51221   ],
      dtype=float32)

In [28]:
# размерность вектора
model.wv["старуха"].shape

(100,)

In [29]:
# похожести слов
model.wv.similarity("тварь", "право")

np.float32(0.07311271)

In [30]:
model.wv.similarity("старуха", "топор")

np.float32(0.40913248)

In [31]:
model.wv.similarity("тварь", "тварь")

np.float32(1.0)

In [32]:
# самые похожие
model.wv.most_similar("старуха")

[('лизавета', 0.4828640818595886),
 ('чиновница', 0.45935046672821045),
 ('близко', 0.4553605616092682),
 ('топор', 0.4091324806213379),
 ('ограбить', 0.40788882970809937),
 ('наклепать', 0.3985449969768524),
 ('ранить', 0.3844590187072754),
 ('убийство', 0.3579655587673187),
 ('грабёж', 0.34470123052597046),
 ('посетитель', 0.34233173727989197)]

In [33]:
model.wv.most_similar("лестница")

[('ступенька', 0.5347699522972107),
 ('передний', 0.49847960472106934),
 ('прихожая', 0.4941595196723938),
 ('узенький', 0.4816994071006775),
 ('всходить', 0.48070499300956726),
 ('вниз', 0.4620290696620941),
 ('цыпочки', 0.441153883934021),
 ('ступень', 0.4395635426044464),
 ('смутно', 0.42081671953201294),
 ('спускаться', 0.419380784034729)]

In [34]:
# арифметика
model.wv.most_similar(positive=["раскольников"], negative=["тварь"])[:10]

[('несколько', 0.3496902883052826),
 ('остановиться', 0.3373570740222931),
 ('пристально', 0.324497252702713),
 ('хорошенько', 0.29947349429130554),
 ('скрываться', 0.29494425654411316),
 ('испугаться', 0.27746421098709106),
 ('направиться', 0.2767505645751953),
 ('промолвить', 0.27071356773376465),
 ('возле', 0.2670804560184479),
 ('провожать', 0.2646567225456238)]

## 4. Как дообучить модель?

Ради чистоты эксперимента сохраним текущую модель и заново подгрузим её.

In [35]:
model_path = "./our_w2v.model"
model.save(model_path)

In [36]:
our_model = Word2Vec.load(model_path)

Подгрузим другое произведение и сделаем для него предобработку.

In [37]:
url = "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/vector-school/SkazkaOCareSaltane.txt"

resp = requests.get(url)
text2 = resp.text

# Последние 500 символов. Аккуратно! Спойлеры!
print(text2[-500:])

 узнает...
В нем взыграло ретивое!
"Что я вижу? что такое?
Как!" - и дух в нем занялся...
Царь слезами залился,
Обнимает он царицу,
И сынка, и молодицу,



И садятся все за стол;
И веселый пир пошел.
А ткачиха с поварихой,
С сватьей бабой Бабарихой
Разбежались по углам;
Их нашли насилу там.
Тут во всем они признались,
Повинились, разрыдались;
Царь для радости такой
Отпустил всех трех домой.
День прошел - царя Салтана
Уложили спать вполпьяна.
Я там был; мед, пиво пил -
И усы лишь обмочил.

1831




Предобработка.

In [38]:
text2 = text2.lower()
sents2 = sent_tokenize(text2)

sents_tokenize2 = [tokenizer.tokenize(sent) for sent in sents2]
sents_tokenize2 = [
    [morph.normal_forms(word)[0] for word in text_cur if word not in stopwords_ru]
    for text_cur in sents_tokenize2
]

In [39]:
sents_tokenize2[10]

['сени', 'выйти', 'царь', 'отец']

In [40]:
len(sents_tokenize2)

254

Дополняем модель.

In [41]:
# обновили словарь
our_model.build_vocab(sents_tokenize2, update=True)

# дообучили
our_model.train(sents_tokenize2, total_examples=our_model.corpus_count, epochs=100)

(177334, 269000)

In [42]:
"ядро" in model.wv.key_to_index

False

In [43]:
"ядро" in our_model.wv.key_to_index

True

In [44]:
our_model.wv.most_similar("ядро")

[('изумруд', 0.7294173836708069),
 ('грызть', 0.6789456605911255),
 ('скатерть', 0.6263769865036011),
 ('орешек', 0.6054326891899109),
 ('чистый', 0.593830406665802),
 ('золотой', 0.5805500745773315),
 ('складка', 0.5451228618621826),
 ('траур', 0.5390905737876892),
 ('научить', 0.5382561087608337),
 ('журчать', 0.5313487648963928)]

Пример со старым словом.

In [45]:
our_model.wv.most_similar("сын")

[('губерния', 0.5306903719902039),
 ('посетить', 0.49499791860580444),
 ('обеспечить', 0.49007394909858704),
 ('наущение', 0.4845471680164337),
 ('благословить', 0.46549275517463684),
 ('приём', 0.4615950584411621),
 ('расспрос', 0.4572676718235016),
 ('родственница', 0.44175639748573303),
 ('паспорт', 0.43840712308883667),
 ('адвокатский', 0.43591806292533875)]

In [46]:
model.wv.most_similar("сын")

[('наущение', 0.469460666179657),
 ('приём', 0.46497154235839844),
 ('посетить', 0.4524017870426178),
 ('детство', 0.44039663672447205),
 ('губерния', 0.4400571882724762),
 ('раздражать', 0.4398869574069977),
 ('соглашаться', 0.43669530749320984),
 ('секрет', 0.4330303370952606),
 ('интерес', 0.4328412115573883),
 ('следить', 0.4324146509170532)]